# 👂 Auditory System: Structure & Function – Interactive Tutor

A self-paced mini-course built from the lecture deck **"Auditory System – Structure and Function"** and the **Chapter 6 practice exercises (6-1 to 6-27)**. Every exercise has a model answer, a hint, and a keyword-based self-check.

### How to use this notebook (about 2–3 hours in total)
1. Run **STEP 1** and **STEP 2** below (click ▶ on each cell, in order). Or use *Runtime → Run all*.
2. Work through **Modules 1–8** in order. Each module follows the same loop:
   **📖 Read → 🧪 Play with the lab → 📝 Quiz → ✍️ Write your own answer → ✔ Check → 👁 Model answer**
3. Finish with the **flashcards**, the **mixed final exam**, your **scoreboard**, and the printable **answer key**.

### Course map
| Module | Topic | Worksheet exercises |
|---|---|---|
| 1 | Temporal bone | 6-1, 6-2 |
| 2 | Overview & external ear | 6-4, 6-5 |
| 3 | Tympanic membrane | 6-6, 6-7, 6-8 |
| 4 | Middle ear cavity (+ revisit the 6-3 figure) | 6-3a–g, 6-9 → 6-16 |
| 5 | Impedance matching | 6-17, 6-18 |
| 6 | Inner ear, cochlea, organ of Corti | 6-19 → 6-22 |
| 7 | Tonotopy & the traveling wave | 6-23, 6-24 |
| 8 | Central pathways & the audiogram | 6-25 → 6-27 |

> 🖼️ **Figure questions** (6-3, 6-6, 6-19, 6-21) refer to pictures printed in your worksheet, so keep it beside you; the answers here tell you exactly what to label.
>
> 🔎 Items marked **“beyond the slides”** come from standard anatomy knowledge, not the lecture deck – good for understanding, but check what your instructor expects.
>
> 📚 Exercise prompts are paraphrased from the Chapter 6 worksheet of *Foundations of Speech and Hearing: Anatomy and Physiology* (Hoit & Weismer, Plural Publishing). Please keep this notebook for personal study rather than posting it publicly.

In [ ]:
#@title ▶ STEP 1 – Run this cell first (loads the tutoring engine) { display-mode: "form" }
import re, random, html as _html
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, HTML, clear_output

PROGRESS = {"quiz": {}, "practice": set(), "cards_seen": 0}

_COLORS = {
    "blue":  ("#eef5ff", "#2b6cb0"),
    "green": ("#edf9f0", "#2f855a"),
    "red":   ("#fff1f0", "#c53030"),
    "gold":  ("#fff9e6", "#b7791f"),
    "gray":  ("#f4f5f7", "#718096"),
}

def box(body, color="blue"):
    """Inline-styled box so it looks right in light AND dark Colab themes."""
    bg, border = _COLORS[color]
    return (f"<div style='background:{bg};color:#1a202c;border-left:6px solid {border};"
            f"border-radius:8px;padding:10px 14px;margin:6px 0;font-family:sans-serif;"
            f"font-size:14.5px;line-height:1.5'>{body}</div>")

# ----------------------------------------------------------------------------
# 1. PRACTICE EXERCISES  (type an answer -> keyword check -> model answer)
# ----------------------------------------------------------------------------
def _practice_one(qid):
    q = EX[qid]
    head = W.HTML(box(f"<b>Exercise {qid}</b><br>{q['prompt']}"
                      + (f"<br><i>🖼 {q['figure']}</i>" if q.get("figure") else ""), "blue"))
    ta = W.Textarea(placeholder="Type your answer here first – then click Check. (Short phrases are fine.)",
                    layout=W.Layout(width="95%", height="95px"))
    b_check = W.Button(description="✔ Check my answer", button_style="primary",
                       layout=W.Layout(width="190px"))
    b_hint = W.Button(description="💡 Hint", layout=W.Layout(width="100px"))
    b_show = W.Button(description="👁 Model answer", button_style="info",
                      layout=W.Layout(width="160px"))
    out = W.Output()

    def on_check(_):
        out.clear_output()
        text = ta.value.lower()
        with out:
            if len(text.strip()) < 3:
                display(HTML(box("Write an attempt first – even a rough one helps you learn more than reading the answer.", "gold")))
                return
            hits, miss = [], []
            for label, pats in q["keys"]:
                (hits if any(re.search(p, text) for p in pats) else miss).append(label)
            pct = round(100 * len(hits) / len(q["keys"]))
            if pct >= 60:
                PROGRESS["practice"].add(qid)
            colour = "green" if pct >= 75 else ("gold" if pct >= 40 else "red")
            body = f"<b>Key ideas found: {len(hits)} of {len(q['keys'])} ({pct}%)</b><br>"
            body += "".join(f"✅ {h}<br>" for h in hits)
            body += "".join(f"⬜ <i>Missing:</i> {m}<br>" for m in miss)
            body += ("<br><small>This is a keyword check, not a grade: a correct answer worded "
                     "differently may be marked missing. Compare with the model answer.</small>")
            display(HTML(box(body, colour)))

    def on_hint(_):
        out.clear_output()
        with out:
            display(HTML(box("💡 <b>Hint:</b> " + q["hint"], "gold")))

    def on_show(_):
        out.clear_output()
        with out:
            display(HTML(box("<b>Model answer</b><br>" + q["answer"], "green")))

    b_check.on_click(on_check); b_hint.on_click(on_hint); b_show.on_click(on_show)
    return W.VBox([head, ta, W.HBox([b_check, b_hint, b_show]), out])

def practice_set(ids):
    """Show several exercises, one after another."""
    display(W.VBox([_practice_one(i) for i in ids]))

# ----------------------------------------------------------------------------
# 2. QUIZZES (multiple choice, instant feedback)
# ----------------------------------------------------------------------------
def _quiz_widget(items, title, key):
    radios, blocks = [], []
    for i, (q, opts, ans, why) in enumerate(items, 1):
        r = W.RadioButtons(options=[(o, j) for j, o in enumerate(opts)], value=None,
                           layout=W.Layout(width="95%"))
        radios.append(r)
        blocks.append(W.VBox([W.HTML(f"<div style='font-family:sans-serif;font-size:14.5px;"
                                     f"margin-top:8px'><b>{i}. {q}</b></div>"), r]))
    btn = W.Button(description="Check my answers", button_style="success",
                   layout=W.Layout(width="200px"))
    out = W.Output()

    def on_click(_):
        out.clear_output()
        score, lines = 0, []
        for i, ((q, opts, ans, why), r) in enumerate(zip(items, radios), 1):
            if r.value is None:
                lines.append(box(f"<b>{i}.</b> ⏭ Not answered. Correct answer: <b>{_html.escape(opts[ans])}</b><br>{why}", "gray"))
            elif r.value == ans:
                score += 1
                lines.append(box(f"<b>{i}.</b> ✅ Correct! {why}", "green"))
            else:
                lines.append(box(f"<b>{i}.</b> ❌ You chose <i>{_html.escape(opts[r.value])}</i>. "
                                 f"Correct: <b>{_html.escape(opts[ans])}</b><br>{why}", "red"))
        PROGRESS["quiz"][key] = (score, len(items))
        pct = round(100 * score / len(items))
        msg = ("🌟 Excellent!" if pct >= 90 else "👍 Solid – review the red ones." if pct >= 70
               else "📖 Re-read the lesson, then try again (click Check again after changing answers).")
        with out:
            display(HTML(box(f"<b>Score: {score}/{len(items)} ({pct}%)</b> – {msg}", "blue")))
            for l in lines:
                display(HTML(l))

    btn.on_click(on_click)
    display(W.VBox([W.HTML(f"<h3 style='font-family:sans-serif'>📝 {title}</h3>")] + blocks + [btn, out]))

def quiz(module):
    _quiz_widget(QUIZ[module], f"Quick quiz – {MODULE_TITLES[module]}", module)

def final_exam(n=20, seed=None):
    """Random mixed quiz across every module."""
    pool = [item for m in QUIZ for item in QUIZ[m]]
    rnd = random.Random(seed)
    items = rnd.sample(pool, min(n, len(pool)))
    _quiz_widget(items, f"Final mixed exam ({len(items)} questions)", "FINAL")

# ----------------------------------------------------------------------------
# 3. FLASHCARDS
# ----------------------------------------------------------------------------
def flashcards(deck=None, shuffle=True):
    cards = [c for c in CARDS if deck is None or c[2] == deck]
    if shuffle:
        random.shuffle(cards)
    st = {"i": 0, "back": False}
    view = W.HTML()
    def draw():
        term, definition, d = cards[st["i"]]
        if st["back"]:
            body = f"<div style='font-size:13px;color:#555'>{d}</div><b style='font-size:17px'>{term}</b><br><br>{definition}"
            col = "green"
        else:
            body = (f"<div style='font-size:13px;color:#555'>{d} · card {st['i']+1}/{len(cards)}</div>"
                    f"<b style='font-size:19px'>{term}</b><br><i>Say the answer out loud, then click Flip.</i>")
            col = "blue"
        view.value = box(body, col)
    def flip(_):
        st["back"] = not st["back"]
        if st["back"]:
            PROGRESS["cards_seen"] += 1
        draw()
    def nxt(_):
        st["i"] = (st["i"] + 1) % len(cards); st["back"] = False; draw()
    def prv(_):
        st["i"] = (st["i"] - 1) % len(cards); st["back"] = False; draw()
    bs = [W.Button(description=t, layout=W.Layout(width="110px")) for t in ("◀ Back", "🔄 Flip", "Next ▶")]
    bs[0].on_click(prv); bs[1].on_click(flip); bs[2].on_click(nxt)
    draw()
    display(W.VBox([view, W.HBox(bs)]))

# ----------------------------------------------------------------------------
# 4. PROGRESS + ANSWER KEY
# ----------------------------------------------------------------------------
def scoreboard():
    rows = ""
    for m, title in MODULE_TITLES.items():
        s = PROGRESS["quiz"].get(m)
        rows += f"<tr><td>{title}</td><td>{'%d / %d' % s if s else '—'}</td></tr>"
    f = PROGRESS["quiz"].get("FINAL")
    rows += f"<tr><td><b>Final mixed exam</b></td><td>{'%d / %d' % f if f else '—'}</td></tr>"
    body = (f"<b>📊 Your progress</b><table style='margin-top:6px;border-collapse:collapse'>{rows}</table>"
            f"<br>Practice exercises answered with ≥60% key ideas: <b>{len(PROGRESS['practice'])} / {len(EX)}</b>"
            f"<br>Flashcards flipped: <b>{PROGRESS['cards_seen']}</b>")
    display(HTML(box(body, "blue")))

def answer_key():
    """Print every worksheet exercise with its model answer (good for a PDF/print-out)."""
    parts = ["<div style='font-family:sans-serif;font-size:14px;color:#111;background:#fff;padding:10px'>",
             "<h2>Chapter 6 – Auditory System: Structure &amp; Function – Model Answer Key</h2>"]
    for qid, q in EX.items():
        parts.append(f"<p><b>{qid}.</b> {q['prompt']}<br><span style='color:#2f855a'>{q['answer']}</span></p><hr>")
    parts.append("</div>")
    display(HTML("".join(parts)))

# ----------------------------------------------------------------------------
# 5. INTERACTIVE LABS
# ----------------------------------------------------------------------------
def ear_canal_lab():
    """Quarter-wavelength resonance of the external auditory meatus."""
    def draw(length_cm):
        c = 34400.0                                   # speed of sound in air, cm/s
        f0 = c / (4 * length_cm)                      # tube closed at one end
        f = np.logspace(np.log10(100), np.log10(10000), 400)
        r = f / f0; Q = 4.0
        gain = 20 * np.log10(1 / np.sqrt((1 - r**2) ** 2 + (r / Q) ** 2))
        fig, ax = plt.subplots(figsize=(7, 3.4))
        ax.semilogx(f, gain, lw=2.5, color="#2b6cb0")
        ax.axvline(f0, color="#c53030", ls="--")
        ax.axvspan(3000, 3500, color="#f6ad55", alpha=.3, label="3000–3500 Hz (typical adult)")
        ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Relative boost (dB)")
        ax.set_title(f"Canal length {length_cm:.1f} cm → lowest resonance ≈ {f0:,.0f} Hz  (schematic)")
        ax.set_xlim(100, 10000); ax.set_ylim(-15, 18); ax.grid(alpha=.3); ax.legend(loc="lower left")
        plt.show()
        print(f"f = c / 4L = 34,400 cm/s ÷ (4 × {length_cm:.1f} cm) = {f0:,.0f} Hz")
    W.interact(draw, length_cm=W.FloatSlider(value=2.5, min=1.0, max=4.0, step=0.1,
                                             description="Length (cm)", continuous_update=False,
                                             style={"description_width": "90px"}))

def middle_ear_gain_lab():
    """Why the middle ear fixes the air-to-fluid impedance mismatch."""
    def draw(tm_area=55.0, footplate_area=3.2, lever=1.3):
        area_ratio = tm_area / footplate_area
        area_db = 20 * np.log10(area_ratio)
        lever_db = 20 * np.log10(lever)
        total = area_db + lever_db
        fig, ax = plt.subplots(figsize=(6.5, 3))
        ax.bar(["Area ratio\n(TM ÷ footplate)", "Lever ratio\n(malleus ÷ incus)", "Total\n(theoretical)"],
               [area_db, lever_db, total], color=["#2b6cb0", "#38a169", "#d69e2e"])
        ax.axhspan(20, 25, color="#c53030", alpha=.15, label="≈20–25 dB measured in real ears")
        ax.set_ylabel("Pressure gain (dB)"); ax.legend(); ax.set_ylim(0, 35); ax.grid(axis="y", alpha=.3)
        for i, v in enumerate([area_db, lever_db, total]):
            ax.text(i, v + .6, f"{v:.1f} dB", ha="center", fontweight="bold")
        plt.show()
        print(f"Area ratio {area_ratio:.1f}:1  →  {area_db:.1f} dB      Lever ratio {lever:.2f}:1  →  {lever_db:.1f} dB")
        print("Why real ears show a little less than the theoretical total: the eardrum does not move as a perfectly\n"
              "rigid piston, and some energy is lost in the joints, ligaments and at high/low frequencies.")
        if tm_area <= footplate_area:
            print("⚠️ With no area advantage the middle ear loses its main source of gain – think about what that means clinically.")
    st = {"description_width": "130px"}
    W.interact(draw,
               tm_area=W.FloatSlider(value=55, min=5, max=90, step=1, description="TM area (mm²)", style=st, continuous_update=False),
               footplate_area=W.FloatSlider(value=3.2, min=1, max=10, step=.1, description="Footplate (mm²)", style=st, continuous_update=False),
               lever=W.FloatSlider(value=1.3, min=1.0, max=2.0, step=.05, description="Lever ratio", style=st, continuous_update=False))

def tonotopy_lab():
    """Where does a tone peak along the basilar membrane? (Greenwood map, schematic wave)"""
    def draw(freq=1000.0, phase=0.0):
        A, a, k = 165.4, 2.1, 0.88                       # Greenwood (1990) human constants
        x_from_apex = np.log10(freq / A + k) / a         # 0 = apex, 1 = base
        p0 = float(np.clip(1 - x_from_apex, 0.02, 0.99))  # 0 = base (oval window), 1 = apex
        p = np.linspace(0, 1, 600)
        rise = (p / p0) ** 1.6
        fall = np.exp(-(((p - p0) / 0.07) ** 2))
        env = np.where(p <= p0, rise, fall)
        wave = env * np.cos(2 * np.pi * (5 * np.minimum(p, p0 * 1.1) / p0) - phase)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.2), gridspec_kw={"height_ratios": [1, 1.5]})
        width = 0.1 + 0.4 * p
        ax1.fill_between(p, -width, width, color="#d6bcfa", alpha=.8)
        for xx in np.linspace(0, 1, 40):
            ax1.axvline(xx, color="white", lw=.6, alpha=.5)
        ax1.plot([p0], [0], "v", ms=14, color="#c53030")
        ax1.text(0.0, .62, "BASE\n(oval window)\nnarrow · STIFF\nHIGH freq", va="bottom", fontsize=9, fontweight="bold")
        ax1.text(1.0, .62, "APEX\n(helicotrema)\nwide · FLOPPY\nLOW freq", va="bottom", ha="right", fontsize=9, fontweight="bold")
        ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-.6, 1.9); ax1.axis("off")
        ax1.set_title(f"Basilar membrane, 'rolled out' – a {freq:,.0f} Hz tone peaks at the red marker", fontsize=10)
        ax2.plot(p, env, "k--", lw=1); ax2.plot(p, -env, "k--", lw=1)
        ax2.plot(p, wave, color="#2b6cb0", lw=2); ax2.axvline(p0, color="#c53030", ls=":")
        ax2.set_xlim(-0.02, 1.02); ax2.set_ylim(-1.3, 1.3)
        ax2.set_xlabel("Distance from base → apex (fraction of BM length)")
        ax2.set_ylabel("Displacement (schematic)"); ax2.set_yticks([])
        plt.tight_layout(); plt.show()
        print(f"Peak ≈ {p0*100:.0f}% of the way from base to apex (≈ {p0*35:.0f} mm of a typical ~35 mm human BM).")
        print("Drag 'phase' to watch the wave travel. Note the slow build-up and abrupt drop-off after the peak.")
    W.interact(draw,
               freq=W.FloatLogSlider(value=1000, base=10, min=np.log10(20), max=np.log10(20000), step=0.02,
                                     description="Frequency (Hz)", style={"description_width": "110px"}, continuous_update=False),
               phase=W.FloatSlider(value=0, min=0, max=2 * np.pi, step=0.2, description="Wave phase",
                                   style={"description_width": "110px"}))

AUD_FREQS = [250, 500, 1000, 2000, 4000, 8000]
PRESETS = {
    "Normal hearing":              [10, 10, 10, 10, 15, 15],
    "Low-frequency loss (rising)": [55, 50, 35, 20, 10, 10],
    "High-frequency loss (sloping)": [10, 10, 15, 35, 60, 70],
    "Flat mild loss":              [35, 35, 35, 35, 35, 35],
    "Mid-frequency ('cookie-bite')": [15, 30, 50, 50, 30, 20],
}

def audiogram_lab():
    """Build an audiogram and have it classified."""
    sl = [W.IntSlider(value=v, min=-10, max=110, step=5, description=f"{f} Hz", continuous_update=False,
                      style={"description_width": "70px"}) for f, v in zip(AUD_FREQS, PRESETS["Low-frequency loss (rising)"])]
    dd = W.Dropdown(options=list(PRESETS), value="Low-frequency loss (rising)", description="Preset:")
    out = W.Output()
    def redraw(*_):
        vals = [s.value for s in sl]
        out.clear_output(wait=True)
        with out:
            fig, ax = plt.subplots(figsize=(6.6, 4.6))
            ax.axhspan(-10, 25, color="#c6f6d5", alpha=.6, label="Normal range (≤ 25 dB HL)")
            ax.plot(AUD_FREQS, vals, "o-", color="#c53030", ms=9, lw=2, label="Threshold (right ear)")
            ax.set_xscale("log"); ax.set_xticks(AUD_FREQS); ax.set_xticklabels([str(f) for f in AUD_FREQS])
            ax.minorticks_off(); ax.xaxis.tick_top(); ax.xaxis.set_label_position("top")
            ax.set_ylim(120, -10); ax.set_yticks(range(-10, 121, 10))
            ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Hearing level (dB HL)  – lower on the graph = worse hearing")
            ax.grid(alpha=.35); ax.legend(loc="lower right", fontsize=8)
            plt.show()
            lo, hi = np.mean(vals[:2]), np.mean(vals[4:])
            pta = np.mean(vals[1:4])
            degree = ("normal" if pta <= 25 else "mild" if pta <= 40 else "moderate" if pta <= 55 else
                      "moderately-severe" if pta <= 70 else "severe" if pta <= 90 else "profound")
            if lo - hi >= 15:   shape = "RISING (low-frequency) configuration – worse at 250–500 Hz than at 4000–8000 Hz"
            elif hi - lo >= 15: shape = "SLOPING (high-frequency) configuration – worse at 4000–8000 Hz"
            else:               shape = "roughly FLAT configuration"
            display(HTML(box(f"<b>Shape:</b> {shape}<br><b>Pure-tone average (500, 1000, 2000 Hz):</b> "
                             f"{pta:.0f} dB HL → <b>{degree}</b> (category boundaries vary slightly between clinics)", "blue")))
    def on_preset(ch):
        for s, v in zip(sl, PRESETS[dd.value]):
            s.value = v
    dd.observe(on_preset, names="value")
    for s in sl:
        s.observe(redraw, names="value")
    display(W.VBox([dd, W.HBox([W.VBox(sl[:3]), W.VBox(sl[3:])]), out]))
    redraw()

In [ ]:
#@title ▶ STEP 2 – Run this cell second (loads questions, answers & flashcards) { display-mode: "form" }
MODULE_TITLES = {
    "M1": "Module 1 · Temporal bone",
    "M2": "Module 2 · Overview & external ear",
    "M3": "Module 3 · Tympanic membrane",
    "M4": "Module 4 · Middle ear cavity",
    "M5": "Module 5 · Impedance matching",
    "M6": "Module 6 · Inner ear & cochlea",
    "M7": "Module 7 · Tonotopy & traveling wave",
    "M8": "Module 8 · Central pathways & audiogram",
}

# ---------------------------------------------------------------------------
# PRACTICE EXERCISES  (numbering follows Chapter 6 practice exercises 6-1 … 6-27)
# keys = (label, [regex alternatives])  – matched against the lower-cased answer
# ---------------------------------------------------------------------------
EX = {
"6-1": dict(
    prompt="In one or two sentences, describe the <b>location and shape</b> of the temporal bone and say why it matters for the hearing mechanism.",
    hint="Think: paired bone, side of the skull, odd shape, extends toward the middle of the skull. What does it surround?",
    answer="The temporal bone is a paired, complexly shaped bone forming part of each side of the skull; it has the bony external auditory meatus on its lateral surface and extends inward toward the middle of the skull. It matters because it encases almost all of the peripheral auditory system – the extremely hard <b>petrous</b> portion houses and protects the inner ear. Only the cartilaginous part of the outer ear and the short stretch of auditory-vestibular nerve between the internal auditory meatus and the brainstem lie outside it.",
    keys=[("Paired bone on each side of the skull", [r"each side|both sides|paired|side of (the )?skull|sides of (the )?skull|lateral"]),
          ("Complex / irregular shape", [r"complex|irregular"]),
          ("Extends medially / petrous part toward the middle of the skull", [r"petrous|medial|middle of (the )?skull|extend"]),
          ("Encases / protects the (inner) ear structures", [r"encas|house|contain|protect|inner ear|peripheral auditory|surround"])]),
"6-2": dict(
    prompt="Describe the <b>location</b> of the external and internal auditory meatuses and state the <b>role</b> of each in the auditory mechanism.",
    hint="One opening is on the outside (lateral) surface and leads to the eardrum; the other faces the inside of the skull (medial surface).",
    answer="<b>External auditory meatus:</b> an opening on the <i>lateral</i> surface of the temporal bone (the bony part of the ear canal); it is the channel that conducts sound from the outside to the tympanic membrane.<br><b>Internal auditory meatus:</b> an opening on the <i>medial</i> surface of the petrous temporal bone, facing the inside of the skull; it is the passage through which the auditory-vestibular nerve (CN VIII, and the facial nerve, CN VII) travels between the inner ear and the brainstem.",
    keys=[("External meatus on the lateral surface", [r"lateral|outside|outer"]),
          ("External meatus leads sound to the tympanic membrane (ear canal)", [r"tympanic|ear ?drum|canal|conduct|channel|funnel"]),
          ("Internal meatus on the medial surface (inside the skull)", [r"medial|inside|petrous|inner"]),
          ("Internal meatus carries nerve(s) VII/VIII to the brainstem", [r"nerve|viii|vii|facial|vestibulocochlear|auditory-vestibular|brain ?stem"])]),
"6-3a": dict(
    prompt="Figure of the peripheral auditory mechanism – <b>(a)</b> What anatomical plane is shown?",
    figure="Use the figure printed in your worksheet.",
    hint="The ear is cut so you see outer, middle and inner ear side-by-side, as if the head was sliced ear-to-ear.",
    answer="A <b>coronal</b> (frontal) section – the head is sliced vertically from ear to ear, so external, middle and inner ear can be seen together (see slide 7).",
    keys=[("Coronal / frontal plane", [r"coronal|frontal"])]),
"6-3b": dict(
    prompt="Figure – <b>(b)</b> Indicate where the two middle-ear muscles attach to the ossicles (label each arrow).",
    figure="Draw the arrows on your worksheet figure.",
    hint="Tensor tympani pulls on the bone attached to the eardrum; stapedius pulls on the smallest bone.",
    answer="<b>Tensor tympani</b> → attaches to the <b>handle (manubrium) of the malleus</b> (CN V).<br><b>Stapedius</b> → attaches to the <b>stapes</b> (neck/head region, near the footplate end) (CN VII); it comes from the posterior wall of the cavity.",
    keys=[("Tensor tympani → malleus", [r"tensor.*malle|malle.*tensor"]),
          ("Stapedius → stapes", [r"stapedius.*stape|stape.*stapedius"])]),
"6-3c": dict(
    prompt="Figure – <b>(c)</b> Show the boundary between the outer and middle ear, and the boundary between the middle and inner ear.",
    figure="Draw the lines on your worksheet figure.",
    hint="Outer/middle boundary is a membrane (not vertical – it is tilted). Middle/inner boundary is the bony wall with two windows.",
    answer="<b>Outer | middle:</b> the <b>tympanic membrane</b> (drawn as a slanting line – the boundary is more diagonal than vertical).<br><b>Middle | inner:</b> the medial wall of the middle-ear cavity, i.e. the bony labyrinth wall with the <b>oval window</b> (stapes footplate) and <b>round window</b>.",
    keys=[("Outer/middle = tympanic membrane", [r"tympanic|ear ?drum|membrane"]),
          ("Middle/inner = oval & round windows / medial bony wall", [r"oval|round|window|medial|promontory|labyrinth"])]),
"6-3d": dict(
    prompt="Figure – <b>(d)</b> Name and describe the ossicular joints of the middle-ear cavity.",
    hint="Both joints are named with 'incudo-'. One links incus to malleus, the other incus to stapes.",
    answer="<b>Incudomalleolar joint</b> – between the head of the malleus and the body of the incus (saddle-shaped synovial joint).<br><b>Incudostapedial joint</b> – between the end (lenticular process) of the long process of the incus and the head of the stapes (ball-and-socket-type synovial joint).<br>The incus is the middle ossicle and forms joints with both of the others.",
    keys=[("Incudomalleolar (incus–malleus) joint", [r"incudo-?malle|incus.{0,20}malleus|malleus.{0,20}incus"]),
          ("Incudostapedial (incus–stapes) joint", [r"incudo-?staped|incus.{0,20}stapes|stapes.{0,20}incus"]),
          ("Incus in the middle / links the other two", [r"middle|link|connect|between|joint"])]),
"6-3e": dict(
    prompt="Figure – <b>(e)</b> Identify the location of the bony auditory tube.",
    figure="Point to it on your worksheet figure.",
    hint="The auditory tube leaves the middle-ear cavity on the front (anterior) wall heading toward the throat.",
    answer="The bony portion of the auditory (Eustachian) tube opens from the <b>anterior wall</b> of the middle-ear cavity, in the temporal bone, and runs forward and downward toward the nasopharynx (its tympanic end is the always-open end; the cartilaginous part continues to the throat).",
    keys=[("Anterior wall of the middle-ear cavity", [r"anterior|front"]),
          ("Runs toward the nasopharynx / throat", [r"naso|throat|pharyn|toward"]),
          ("In the temporal bone / tympanic end", [r"temporal|bone|bony|tympanic|middle ear"])]),
"6-3f": dict(
    prompt="Figure – <b>(f)</b> Which nerves are shown entering the internal auditory meatus?",
    hint="Two cranial nerves travel together here: one for hearing + balance, one for facial muscles.",
    answer="The <b>vestibulocochlear (auditory-vestibular) nerve, CN VIII</b> (cochlear + vestibular divisions) and the <b>facial nerve, CN VII</b>.",
    keys=[("CN VIII – vestibulocochlear / auditory-vestibular", [r"viii|8|vestibulo|auditory|cochlear"]),
          ("CN VII – facial", [r"vii|facial|7\b"])]),
"6-3g": dict(
    prompt="Figure – <b>(g)</b> Point out the location of the cochlea, vestibule and semicircular canals.",
    figure="Label them on your worksheet figure.",
    hint="Slide 17 gives the order from back to front.",
    answer="From <b>back to front</b>: <b>semicircular canals</b> (posterior/superior, three looping canals) → <b>vestibule</b> (central chamber) → <b>cochlea</b> (the snail-shaped spiral at the front/medial end).",
    keys=[("Semicircular canals at the back", [r"semi.?circular|canal"]),
          ("Vestibule in the middle", [r"vestibule"]),
          ("Cochlea at the front / spiral", [r"cochlea|spiral|snail"])]),
"6-4": dict(
    prompt="List the components of the peripheral auditory mechanism under <b>Conductive</b> and <b>Sensorineural</b>.",
    hint="Conductive = everything that carries sound mechanically up to the oval window. Sensorineural = the transducer plus its nerve.",
    answer="<b>Conductive:</b> pinna; external auditory meatus; tympanic membrane; middle-ear cavity with the three ossicles (malleus, incus, stapes) and middle-ear muscles/ligaments.<br><b>Sensorineural:</b> cochlea (organ of Corti with its hair cells) and the auditory nerve (cochlear division of CN VIII).",
    keys=[("Pinna / external ear canal", [r"pinna|auricle|meatus|ear canal|external ear"]),
          ("Tympanic membrane", [r"tympanic|ear ?drum"]),
          ("Ossicles / middle ear", [r"ossicle|malleus|incus|stapes|middle ear"]),
          ("Cochlea (hair cells / organ of Corti)", [r"cochlea|organ of corti|hair cell"]),
          ("Auditory nerve (CN VIII)", [r"nerve|viii"])]),
"6-5": dict(
    prompt="What is the relationship between the <b>length and shape</b> of the external auditory meatus and its <b>function</b> in general auditory health and in hearing?",
    hint="Length → resonance (quarter-wavelength rule). Shape/slope/glands → protection and self-cleaning.",
    answer="<b>Hearing:</b> the canal is a tube ~2.5 cm long, open at one end and closed at the other (eardrum). By the ¼-wavelength rule its lowest resonance falls at about 3000–3500 Hz, which boosts the sound reaching the eardrum in that range.<br><b>Health:</b> the bend (kink) toward the back of the head helps protect the eardrum from foreign objects; glands in the cartilaginous part secrete cerumen that lubricates the walls and traps/blocks material and bacteria; and the downward slope toward the concha lets sloughed-off dead cells move out, away from the eardrum.",
    keys=[("~2.5 cm tube, closed at one end", [r"2\.5|closed|one end"]),
          ("¼-wavelength resonance ≈ 3000–3500 Hz", [r"resonan|quarter|1/4|¼|3000|3500|3 ?khz"]),
          ("Bend/kink protects the eardrum", [r"bend|kink|curve|foreign|protect"]),
          ("Cerumen (wax) protects/lubricates", [r"cerumen|wax|lubric|bacteria"]),
          ("Slope lets dead cells drain outward", [r"slope|downhill|sloughed|dead cells|drain|clean"])]),
"6-6": dict(
    prompt="Otoscopic view of the tympanic membrane: label at least <b>three landmarks</b> with a one-sentence note on the significance of each, and say <b>which ear</b> is shown and how you know.",
    figure="Use the otoscopic photo in your worksheet.",
    hint="Look for: the handle of the malleus (a streak), the bright triangle of reflected light, the small floppy upper part.",
    answer="<b>Handle (manubrium) of the malleus</b> – visible through the translucent drum; it is attached to the membrane and pulls it into its cone shape, transmitting vibration to the ossicular chain.<br><b>Umbo</b> – the tip of the malleus handle at the centre of the drum; the point of maximal inward displacement.<br><b>Cone of light</b> – bright reflection toward the lower front of the drum; its presence and position suggest a normal, correctly tilted, healthy membrane.<br><b>Pars flaccida</b> – the small, slack upper region (only epithelial layers); <b>pars tensa</b> – the larger, taut part with the fibrous layer.<br><b>Which ear:</b> in a <i>right</i> ear the malleus handle runs from about 1 o'clock down to the umbo and the cone of light sits at about 5 o'clock (in a left ear: ~11 and ~7 o'clock). In the worksheet photo the handle slants up to the right with the light reflex low and right of centre, so it is best read as a <b>right ear</b> – confirm with your instructor.",
    keys=[("Handle of malleus (attached to membrane)", [r"malleus|handle|manubrium"]),
          ("Umbo", [r"umbo"]),
          ("Cone of light / light reflex", [r"cone|light"]),
          ("Pars flaccida / pars tensa", [r"flaccida|tensa"]),
          ("Names the ear (right) with a reason", [r"right|left"])]),
"6-7": dict(
    prompt="The tympanic membrane has three tissue layers. Which layer is most sensitive to molecule-sized vibrations? Why do you think this is the case?",
    hint="Two layers are epithelial (skin-like / lining). The middle layer is made of something strong, springy and fibre-like.",
    answer="The <b>middle layer – the fibrous, collagenous layer</b> – is exquisitely sensitive to tiny pressure variations. A plausible reason: collagen fibres form a thin, light, stiff-yet-elastic sheet that responds quickly to very small pressure changes and springs back, whereas the two outer epithelial layers (one facing the canal, one facing the middle ear) mainly cover and protect.",
    keys=[("Middle layer", [r"middle"]),
          ("Fibrous / collagen tissue", [r"fibrous|collagen|fiber|fibre"]),
          ("Reason: elastic/stiff/light, responds to tiny pressure changes", [r"elastic|stiff|strong|light|thin|spring|respond|sensitiv|vibrat|pressure"])]),
"6-8": dict(
    prompt="In one sentence, describe the anatomical difference between the <b>pars tensa</b> and <b>pars flaccida</b>.",
    hint="Count the layers in each part.",
    answer="The pars tensa (most of the membrane) has all three layers including the stiff middle fibrous/collagen layer, whereas the small pars flaccida (the slack upper part) has only the two epithelial layers and lacks the fibrous layer.",
    keys=[("Pars tensa has all three layers / fibrous layer", [r"tensa.{0,80}(three|3|fibrous|collagen|all)"]),
          ("Pars flaccida has only the epithelial layers", [r"flaccida.{0,80}(only|two|2|epithel|lack|without|no )|(only|two|2).{0,40}epithel"])]),
"6-9": dict(
    prompt="Fill in: The muscle with its origin in the posterior wall of the tympanic cavity is the ______ and attaches to the ______; the muscle with its origin in a bony canal next to the auditory tube is the ______ and attaches to the ______. Contraction of either muscle has the same general effect, which is ______.",
    hint="Stapedius ↔ stapes. Tensor tympani ↔ the bone attached to the drum. Effect: what happens to the ossicular chain's stiffness and to energy reaching the cochlea?",
    answer="<b>Stapedius</b> (innervated by CN VII) → attaches to the <b>stapes</b>; <b>tensor tympani</b> (CN V) → attaches to the <b>handle of the malleus</b>. Either muscle's contraction <b>stiffens the ossicular chain</b>, reducing the sound energy transmitted to the cochlea. The stapedius is the key effector of the acoustic reflex, protecting the cochlea from extremely intense sound by keeping the footplate from being driven too forcefully into the cochlear fluid.",
    keys=[("Stapedius → stapes", [r"stapedius"]),
          ("Tensor tympani → malleus (handle)", [r"tensor"]),
          ("Effect: stiffens chain / reduces transmission", [r"stiff|reduc|dampen|attenuat|decreas|protect|less"])]),
"6-10": dict(
    prompt="Fill in: The <b>medial wall</b> of the middle-ear cavity is the ______, and the <b>lateral wall</b> is the ______.",
    hint="Medial = faces the inner ear (bony, contains the windows). Lateral = faces the ear canal.",
    answer="Medial wall = the <b>labyrinthine wall</b> (bony wall of the inner ear / otic capsule, with the promontory, oval window and round window). Lateral wall = the <b>membranous wall</b> – the <b>tympanic membrane</b> (plus bony ring around it).",
    keys=[("Medial = labyrinthine / bony inner-ear wall (promontory, windows)", [r"labyrinth|promontory|oval|round|otic|inner"]),
          ("Lateral = membranous wall / tympanic membrane", [r"membran|tympanic|ear ?drum"])]),
"6-11": dict(
    prompt="List the ossicles and state how they are attached from the lateral to the medial wall of the middle-ear cavity. State the joints they form.",
    hint="Lateral → medial: the bone fixed to the eardrum, then the middle bone, then the bone in the oval window.",
    answer="<b>Malleus</b> (lateral; its handle is attached to the tympanic membrane) → <b>incus</b> (middle) → <b>stapes</b> (medial; its oval footplate sits in the oval window of the bony labyrinth, held by the annular ligament).<br>Joints: <b>incudomalleolar</b> (malleus–incus) and <b>incudostapedial</b> (incus–stapes). Ligaments hold the ossicles in place and limit their range of movement.",
    keys=[("Malleus attached to the tympanic membrane", [r"malleus.{0,60}(tympanic|membrane|drum)|(tympanic|membrane|drum).{0,60}malleus"]),
          ("Incus in the middle", [r"incus"]),
          ("Stapes footplate in the oval window", [r"stapes.{0,80}(oval|window|footplate)|(oval|window|footplate).{0,60}stapes"]),
          ("Incudomalleolar & incudostapedial joints", [r"incudo|joint"])]),
"6-12": dict(
    prompt="You are asked to identify each ossicle under a magnifier. What <b>unique features</b> of the malleus, incus and stapes would you look for?",
    hint="Names give hints: hammer, anvil, stirrup. Which one has a long handle? Which has a footplate?",
    answer="<b>Malleus (hammer):</b> a long <i>handle</i> (manubrium) that attaches to the eardrum, plus a rounded head that articulates with the incus (also a neck and small processes).<br><b>Incus (anvil):</b> a thick <i>body</i> with a facet for the malleus head, a short process (anchored by a ligament) and a long process ending in a tiny lenticular process that meets the stapes.<br><b>Stapes (stirrup):</b> the smallest bone in the body – a small head, two arch-like limbs (crura) and an oval <i>footplate</i> that fits the oval window.",
    keys=[("Malleus – handle / head", [r"handle|manubrium|head"]),
          ("Incus – body, short & long process", [r"body|short process|long process|anvil|lenticular"]),
          ("Stapes – footplate / crura / stirrup / smallest", [r"footplate|crura|crus|stirrup|smallest"])]),
"6-13": dict(
    prompt="There are two annular ligaments in the peripheral auditory mechanism. Where are they and what do they secure?",
    hint="One ring holds the eardrum in the bony canal; the other holds the footplate in a window.",
    answer="(1) The <b>annular ligament of the tympanic membrane</b> (the fibrocartilaginous tympanic annulus) at the rim of the eardrum – secures the tympanic membrane to the bony sulcus of the temporal bone at the end of the ear canal.<br>(2) The <b>annular ligament of the stapes</b> – a ring of fibres around the footplate that secures it in the <b>oval window</b> of the bony labyrinth while allowing it to move in and out of the fluid.",
    keys=[("Tympanic membrane annulus – holds eardrum in the bony canal", [r"tympanic|drum|membrane"]),
          ("Stapes annular ligament – footplate in oval window", [r"stapes|footplate|oval"])]),
"6-14": dict(
    prompt="Fill in: The ______ end of the auditory tube is always open and the ______ end is normally closed but can open intermittently. The importance of the auditory tube to the conductive mechanism is ______.",
    hint="Which end faces the ear and which faces the throat? Think of what happens when you swallow or yawn on a plane.",
    answer="The <b>tympanic (middle-ear) end</b> is always open; the <b>pharyngeal (nasopharyngeal) end</b> is normally closed and opens briefly during swallowing, yawning or chewing. Importance: it <b>equalises air pressure</b> in the middle-ear cavity with the outside atmosphere so the tympanic membrane can vibrate freely, and it allows ventilation/drainage of the cavity.",
    keys=[("Tympanic / middle-ear end open", [r"tympanic|middle ear|ear end|ossicle"]),
          ("Pharyngeal / throat end normally closed", [r"pharyn|throat|naso"]),
          ("Equalises pressure on both sides of the eardrum", [r"equal|pressure|ventilat|drain"])]),
"6-15": dict(
    prompt="Fill in: In views looking toward the lateral or medial wall of the middle-ear cavity, the nerve seen running through the bony walls is the ______ nerve.",
    hint="Cranial nerve VII.",
    answer="The <b>facial nerve (CN VII)</b>, which runs in its bony canal along the medial wall above the oval window and down the posterior wall. (A branch of it, the chorda tympani, also crosses the cavity.)",
    keys=[("Facial nerve (VII)", [r"facial|vii|7\b"])]),
"6-16": dict(
    prompt="In the auditory periphery, what does the term <b>promontory</b> refer to?",
    hint="A bulge on the medial wall between the two windows.",
    answer="The <b>promontory</b> is the rounded bony bulge on the medial wall of the middle-ear cavity, formed by the basal turn of the cochlea underneath, lying between the oval window and the round window.",
    keys=[("Bulge/bony prominence on the medial wall", [r"bulge|prominen|bump|bony|medial|wall"]),
          ("Formed by the cochlea (basal turn)", [r"cochlea|basal|turn"]),
          ("Between oval and round windows", [r"oval|round|window|between"])]),
"6-17": dict(
    prompt="In three sentences or fewer, explain the concept of <b>impedance mismatch</b>.",
    hint="Air is easy to push; fluid is not. What happens to most of the energy when sound meets the fluid directly?",
    answer="Impedance is a medium's opposition to being set in motion. Air (low impedance) and cochlear fluid (high impedance) are very different, so if sound in air struck the fluid directly most of the energy would be <b>reflected</b> rather than transmitted (roughly 99 %, a loss of about 30 dB). The middle ear works as an <b>impedance-matching transformer</b> that overcomes this mismatch.",
    keys=[("Air vs fluid have different impedance (low vs high)", [r"air|fluid|differen|low|high"]),
          ("Most energy is reflected / lost", [r"reflect|lost|loss|bounce|little energy|most"]),
          ("Middle ear matches / compensates", [r"middle ear|match|transform|compensat|overcome"])]),
"6-18": dict(
    prompt="Explain briefly how the anatomy of the conductive mechanism produces a gain of roughly 20–25 dB from the tympanic membrane to the stapes footplate.",
    hint="Big membrane vs small footplate (area), plus a lever action of the ossicles. Pressure = force ÷ area.",
    answer="<b>Area ratio</b> (main contributor): the effective vibrating area of the tympanic membrane is many times larger (~17:1) than the stapes footplate, so the same force is concentrated on a small area → pressure rises (≈25 dB by itself).<br><b>Lever action</b>: the malleus handle is longer than the long process of the incus, adding a modest force gain (~1.3:1, a couple of dB).<br>Together these give a theoretical boost somewhat above 25 dB; in real ears the measured gain is about 20–25 dB because the eardrum is not a perfect rigid piston and some energy is lost.",
    keys=[("Area ratio: large TM vs small footplate", [r"area|larger|smaller|surface|concentrat"]),
          ("Pressure = force/area (pressure increases)", [r"pressure|force"]),
          ("Ossicular lever action", [r"lever|ossic|long|handle"])]),
"6-19": dict(
    prompt="Bony labyrinth (otic capsule): identify the major contents with arrows and labels, and write a one-sentence function for each.",
    figure="Label the figure in your worksheet.",
    hint="Three main chambers (balance canals, vestibule, cochlea) plus two windows.",
    answer="<b>Semicircular canals</b> – fluid and hair cells that detect head motion; critical for balance.<br><b>Vestibule</b> – structures that detect head position and motion relative to the body.<br><b>Cochlea</b> – spiral fluid-filled structure with hair cells that detect sound energy.<br><b>Oval window</b> – opening where the stapes footplate delivers vibration into the scala vestibuli.<br><b>Round window</b> – second opening at the end of the scala tympani that lets the incompressible fluid move (pressure relief).",
    keys=[("Semicircular canals – head motion / balance", [r"semi.?circular|canal"]),
          ("Vestibule – head position", [r"vestibule"]),
          ("Cochlea – hearing / sound detection", [r"cochlea"]),
          ("Oval window", [r"oval"]),
          ("Round window", [r"round"])]),
"6-20": dict(
    prompt="List the three ducts of the cochlea, the membranes that separate them, and the duct that contains the organ of Corti.",
    hint="Reissner's membrane and the basilar membrane are the dividers. The organ of Corti sits in the middle duct.",
    answer="<b>Scala vestibuli</b> (top, perilymph), <b>scala media / cochlear duct</b> (middle, endolymph) and <b>scala tympani</b> (bottom, perilymph).<br><b>Reissner's membrane</b> separates scala vestibuli from scala media; the <b>basilar membrane</b> separates scala media from scala tympani.<br>The <b>scala media</b> contains the <b>organ of Corti</b>. (Scala vestibuli and scala tympani connect at the apex, the helicotrema.)",
    keys=[("Scala vestibuli", [r"vestibuli"]),
          ("Scala media (cochlear duct)", [r"media|cochlear duct"]),
          ("Scala tympani", [r"tympani"]),
          ("Reissner's membrane", [r"reissner"]),
          ("Basilar membrane", [r"basilar"])]),
"6-21": dict(
    prompt="What is shown in the cross-section figure? Label the important structures and give one sentence on each one's function/importance.",
    figure="Use the organ-of-Corti figure in your worksheet.",
    hint="Look for one hair cell on one side and three on the other, with a flap lying above them.",
    answer="The figure shows the <b>organ of Corti</b> (in cross-section, within the scala media).<br><b>Basilar membrane</b> – supports the organ of Corti and moves with the traveling wave.<br><b>Inner hair cell (one row)</b> – primary sensory cell for frequency analysis; sends signals via afferent nerve fibres.<br><b>Outer hair cells (three rows)</b> – amplify and sharpen the frequency analysis.<br><b>Stereocilia</b> – hair-like projections whose bending depolarises the hair cell.<br><b>Tectorial membrane</b> – overlying flap coupled to the hair cells' stereocilia (directly to the outer hair cells, via fluid motion for the inner).<br><b>Nerve fibres</b> – emerge from the base of the hair cells; afferent fibres (mainly from inner cells) carry signals to the brain and efferent fibres return to outer cells.<br><b>Supporting cells</b> – give the organ structure.",
    keys=[("Organ of Corti", [r"corti"]),
          ("Basilar membrane", [r"basilar"]),
          ("Inner hair cells (one row)", [r"inner hair|inner cell"]),
          ("Outer hair cells (three rows)", [r"outer hair|outer cell"]),
          ("Tectorial membrane", [r"tectorial"]),
          ("Stereocilia", [r"stereocili|cilia"]),
          ("Nerve fibres (afferent/efferent)", [r"nerve|afferent|efferent|fibre|fiber"])]),
"6-22": dict(
    prompt="What is the <b>functional difference</b> between inner and outer hair cells?",
    hint="One row does the primary frequency analysis; three rows amplify and sharpen.",
    answer="<b>Inner hair cells</b> (one row) perform the <b>primary frequency analysis</b> – they are the main sensory transducers whose basal nerve fibres form the afferent part of the auditory nerve.<br><b>Outer hair cells</b> (three rows) <b>amplify</b> the vibration and <b>sharpen</b> the frequency analysis of the inner hair cells; their nerve fibres form the efferent part (signals from the brain back to the cochlea). Both are arranged tonotopically.",
    keys=[("Inner = primary frequency analysis / afferent", [r"inner.{0,100}(frequency|analysis|afferent|primary|transduc)"]),
          ("Outer = amplification", [r"outer.{0,100}(amplif|boost|gain)|amplif"]),
          ("Outer = sharpen tuning / efferent", [r"sharpen|tuning|efferent|selectiv"])]),
"6-23": dict(
    prompt="Describe the <b>tonotopic organization</b> of the hair cells along the basilar membrane. How does the membrane's stiffness vary from base to apex and how does that relate to tonotopy?",
    hint="Remember: Base = high frequency (narrow, stiff). Apex = low frequency (wide, floppy).",
    answer="Hair cells are arranged by the frequency they respond to best: <b>high frequencies (up to ~20,000 Hz) at the base</b>, progressing to <b>low frequencies (down to ~20 Hz) at the apex</b>. The basilar membrane is <b>narrowest and stiffest at the base</b> and <b>widest and floppiest at the apex</b>, so stiff basal regions vibrate best to high frequencies and floppy apical regions to low frequencies – the mechanical gradient matches the tonotopic map.",
    keys=[("High frequencies at the base", [r"base.{0,60}high|high.{0,60}base"]),
          ("Low frequencies at the apex", [r"apex.{0,60}low|low.{0,60}apex"]),
          ("Base narrow & stiff", [r"stiff.{0,60}base|base.{0,60}(stiff|narrow)|narrow"]),
          ("Apex wide & floppy", [r"apex.{0,60}(wide|floppy|flexible|loose|less stiff)|floppy|wide"])]),
"6-24": dict(
    prompt="What is a <b>traveling wave</b> in the cochlea? How is it related to tonotopic organization?",
    hint="Wave develops over time, starts at the stapes end, and peaks at a frequency-dependent place.",
    answer="When the stapes footplate pushes the cochlear fluid, a wave pattern develops over time along the basilar membrane, travelling from the base toward the apex. The wave's <b>maximum displacement occurs at a place that depends on frequency</b>: high frequencies peak near the stiff base, low frequencies near the floppy apex. The peak deforms the hair cells there, depolarising them and firing the attached nerve fibres – so the place of the peak <i>is</i> the tonotopic code for frequency.",
    keys=[("Wave develops over time along the basilar membrane (base → apex)", [r"over time|travel|base.{0,40}apex|along"]),
          ("Caused by stapes/fluid displacement", [r"stapes|footplate|fluid|oval"]),
          ("Peak location depends on frequency", [r"peak|maxim|place|location"]),
          ("Hair cells deformed → nerve fires", [r"hair cell|depolari|nerve|fire|deform"])]),
"6-25": dict(
    prompt="Explain the anatomical basis of the very precise <b>binaural</b> (two-ear) capabilities of humans.",
    hint="Where does each ear's signal go in the brain? One side only, or both?",
    answer="Input from <b>each ear is represented on both sides</b> of the brain: after entering the CNS at the brainstem (medulla), auditory pathways cross and interconnect at multiple levels on their way to the primary auditory cortex of both temporal lobes, with <b>extensive interactions between the two halves of the CNS</b>. Because neurons can compare timing/level information from both ears, this bilateral wiring supports high-precision binaural abilities such as localising sound. <i>(Beyond the slides: the first such comparisons occur in the superior olivary complex of the brainstem.)</i>",
    keys=[("Each ear represented on both sides of the brain", [r"both (sides|hemispheres)|bilateral|each ear|both ears"]),
          ("Pathways cross / interact at several levels", [r"cross|interact|connect|decussat|multiple|extensive"]),
          ("Comparison of the two ears' signals → localisation/precision", [r"compar|locali|precis|timing|difference"])]),
"6-26": dict(
    prompt="What standard frequencies are tested to create an audiogram?",
    hint="Octave frequencies from 250 up to 8000 Hz (125 Hz is sometimes added).",
    answer="The standard octave frequencies <b>250, 500, 1000, 2000, 4000 and 8000 Hz</b> (often with 125 Hz at the low end, and inter-octave points such as 3000 and 6000 Hz added when needed).",
    keys=[("250 / 500 Hz", [r"250|500"]), ("1000 / 2000 Hz", [r"1000|2000|1 ?k|2 ?k"]), ("4000 / 8000 Hz", [r"4000|8000|4 ?k|8 ?k"])]),
"6-27": dict(
    prompt="Describe an <b>audiogram</b> of someone with a low-frequency hearing loss.",
    hint="Which end of the graph do the thresholds drop on? Think 'rising' or 'ascending'.",
    answer="Thresholds are <b>poorer (higher dB HL, lower on the graph) at the low frequencies</b> (250–500 Hz, e.g., 40–60 dB HL) and <b>improve toward normal at mid and high frequencies</b> (≤ ~25 dB HL at 2000–8000 Hz). The curve therefore <b>rises from left to right</b> – a 'rising' (ascending) configuration, the mirror image of the common high-frequency sloping loss.",
    keys=[("Worse thresholds at low frequencies (250/500 Hz)", [r"low"]),
          ("Better/normal at high frequencies", [r"high|normal|better|improv"]),
          ("Rising / ascending shape", [r"rising|rise|ascend|upward|slop|left to right"])]),
}

# ---------------------------------------------------------------------------
# QUIZZES   (question, [options], index_of_correct, explanation)
# ---------------------------------------------------------------------------
QUIZ = {
"M1": [
 ("Which part of the temporal bone contains the inner-ear structures?",
  ["The squamous part forming the cheek", "The petrous part", "The cartilaginous part", "The external auditory meatus"], 1,
  "The petrous part is extremely hard bone lying medially in the skull; it protects the inner ear."),
 ("Which parts of the peripheral auditory system are NOT encased by the temporal bone?",
  ["Cochlea and vestibule", "Ossicles and middle-ear muscles", "Cartilaginous outer ear and the short nerve segment between the internal meatus and brainstem", "The oval and round windows"], 2,
  "Most of the peripheral system is inside the bone; the exceptions are the cartilaginous pinna/outer canal and the short section of the auditory-vestibular nerve near the brainstem."),
 ("The internal auditory meatus is found on which surface of the temporal bone?",
  ["Lateral surface", "Medial surface", "Inferior surface of the pinna", "Outer surface of the tympanic membrane"], 1,
  "The external meatus opens laterally; the internal meatus opens on the medial surface, facing the brain."),
 ("What is the main function of the external auditory meatus?",
  ["Carry cranial nerves to the brainstem", "Channel sound to the tympanic membrane", "Amplify sound by 60 dB", "Produce endolymph"], 1,
  "The external meatus is the canal that conducts sound to the eardrum."),
 ("Which two cranial nerves travel through the internal auditory meatus?",
  ["I and II", "III and IV", "VII and VIII", "IX and X"], 2,
  "Facial (VII) and vestibulocochlear/auditory-vestibular (VIII)."),
],
"M2": [
 ("Which parts of the peripheral auditory system are air-filled?",
  ["Inner ear only", "External and middle ear", "Middle and inner ear", "All three parts"], 1,
  "External and middle ears contain air; the inner ear is fluid-filled."),
 ("The structures of the pinna help us to...",
  ["Equalise pressure", "Localise sound in the up–down and front–back dimensions", "Make cerumen", "Stiffen the ossicular chain"], 1,
  "Pinna folds modify sound spectra depending on the direction of the source."),
 ("The external auditory meatus is ~2.5 cm long and closed at one end. Its lowest resonance is about...",
  ["300–350 Hz", "1400 Hz", "3000–3500 Hz", "10,000 Hz"], 2,
  "By the ¼-wavelength rule, f ≈ c/4L ≈ 34,400/(4 × 2.5) ≈ 3400 Hz."),
 ("What does cerumen (ear wax) do?",
  ["Lubricates the canal and helps block bacteria/debris from moving toward the eardrum", "Transmits vibration to the ossicles", "Equalises pressure", "Amplifies low frequencies"], 0,
  "Glands in the cartilaginous canal secrete it; it lubricates and protects."),
 ("The walls of the ear canal change from cartilage to bone about...",
  ["Right at the concha", "Halfway to the tympanic membrane", "At the oval window", "Only in the middle-ear cavity"], 1,
  "About halfway in; the bend sits roughly at the cartilage–bone junction."),
 ("Which structure belongs to the SENSORINEURAL mechanism?",
  ["Tympanic membrane", "Stapes", "Organ of Corti", "Pinna"], 2,
  "Cochlea (organ of Corti) and auditory nerve are sensorineural; the rest are conductive."),
],
"M3": [
 ("How many tissue layers does the tympanic membrane have, and which is the middle layer?",
  ["Two; muscular", "Three; fibrous/collagenous", "Three; cartilaginous", "Four; mucous"], 1,
  "Two epithelial layers sandwich a fibrous/collagen layer."),
 ("The pars flaccida differs from the pars tensa because it...",
  ["Has only the epithelial layers", "Is thickest", "Is attached to the stapes", "Has three fibrous layers"], 0,
  "Pars flaccida lacks the middle fibrous layer, so it is slack."),
 ("Which layer of the tympanic membrane is exquisitely sensitive to molecule-size pressure variations?",
  ["Outer epithelial", "Middle fibrous", "Inner epithelial", "All equally"], 1,
  "The middle fibrous/collagen layer."),
 ("In the coronal view, the tympanic membrane is best described as...",
  ["Exactly vertical", "Tilted (more diagonal than vertical)", "Horizontal", "Absent"], 1,
  "The outer/middle ear boundary is more diagonal than vertical."),
 ("The handle of which ossicle attaches to the tympanic membrane?",
  ["Stapes", "Incus", "Malleus", "None"], 2,
  "Malleus – its handle is attached to the eardrum."),
 ("The 'cone of light' seen on otoscopy is...",
  ["A nerve", "A reflection of the otoscope light from the translucent membrane", "The stapes footplate", "A sign of ear wax"], 1,
  "It is a normal light reflex (beyond the slides) – helpful for orienting the ear (right vs left)."),
],
"M4": [
 ("Which is the smallest bone in the body?",
  ["Malleus", "Incus", "Stapes", "Hyoid"], 2,
  "The stapes."),
 ("Which muscle's contraction responds to extremely intense sound (acoustic reflex), and which cranial nerve supplies it?",
  ["Tensor tympani – CN V", "Stapedius – CN VII", "Tensor tympani – CN VII", "Stapedius – CN V"], 1,
  "Stapedius – facial nerve (VII); tensor tympani – trigeminal (V)."),
 ("The three middle-ear chambers from top to bottom are...",
  ["Hypotympanum, mesotympanum, epitympanum", "Epitympanum, mesotympanum, hypotympanum", "Mesotympanum, epitympanum, hypotympanum", "Epitympanum, hypotympanum, mesotympanum"], 1,
  "Epi- (above), meso- (middle), hypo- (below)."),
 ("Which ossicle forms joints with both of the other two?",
  ["Malleus", "Incus", "Stapes", "All equally"], 1,
  "The incus – incudomalleolar and incudostapedial joints."),
 ("The 'promontory' is...",
  ["A muscle", "A bulge on the medial wall of the middle ear caused by the cochlea", "A part of the tympanic membrane", "A cranial nerve"], 1,
  "It lies between the oval and round windows."),
 ("Which end of the auditory tube is normally closed?",
  ["Tympanic end", "Pharyngeal end", "Both ends", "Neither end"], 1,
  "It opens intermittently (swallowing/yawning) to equalise pressure."),
 ("The ossicular chain has a resonant frequency of about...",
  ["200 Hz", "1400 Hz", "8000 Hz", "20,000 Hz"], 1,
  "Slide 14: ~1400 Hz."),
 ("The medial wall of the middle-ear cavity is the...",
  ["Membranous wall (tympanic membrane)", "Labyrinthine wall (bony inner-ear wall)", "Tegmental wall only", "Auditory tube"], 1,
  "It carries the promontory, oval window and round window."),
],
"M5": [
 ("Why is there an impedance mismatch between the outer and inner ear?",
  ["Air has low impedance but cochlear fluid has high impedance", "Fluid has low impedance and air has high impedance", "Both have identical impedance", "Because the ossicles are heavy"], 0,
  "Without the middle ear, most energy would be reflected at the air–fluid interface."),
 ("Which mechanism contributes MOST of the middle-ear pressure gain?",
  ["Lever action of the ossicles", "Area ratio of eardrum to footplate", "Cerumen", "The helicotrema"], 1,
  "The large-to-small area ratio (~17:1) provides about 25 dB alone."),
 ("If the stapes footplate area were equal to the effective eardrum area, the area-ratio gain would be about...",
  ["+25 dB", "+12 dB", "0 dB", "−20 dB"], 2,
  "20·log10(1) = 0 dB."),
 ("Roughly how much gain does the middle ear provide?",
  ["2–5 dB", "20–25 dB", "60–80 dB", "100 dB"], 1,
  "About 20–25 dB measured from the TM to the footplate."),
 ("The ossicular lever advantage comes from the fact that...",
  ["The malleus handle is longer than the incus long process", "The stapes is the largest bone", "The incus is shorter than the stapes", "The tensor tympani lengthens the chain"], 0,
  "A longer lever arm on the malleus side gives a modest force increase."),
],
"M6": [
 ("Which fluid fills the scala media?",
  ["Perilymph", "Endolymph", "Cerebrospinal fluid", "Cerumen"], 1,
  "Scala media – endolymph; scala vestibuli and tympani – perilymph."),
 ("Which membrane separates the scala vestibuli from the scala media?",
  ["Basilar membrane", "Tectorial membrane", "Reissner's membrane", "Tympanic membrane"], 2,
  "Reissner's membrane; the basilar membrane lies below the scala media."),
 ("Where do the scala vestibuli and scala tympani connect?",
  ["At the oval window", "At the helicotrema (apex)", "At the round window", "At the vestibule"], 1,
  "At the apex, through the helicotrema."),
 ("How many turns does the human cochlea make?",
  ["1", "2½", "5", "10"], 1,
  "2½ turns from base to apex."),
 ("The inner hair cells are arranged in...",
  ["One row", "Two rows", "Three rows", "Five rows"], 0,
  "One row inner, three rows outer."),
 ("Which statement about outer hair cells is correct?",
  ["They perform primary frequency analysis", "They amplify and sharpen the frequency analysis", "They produce endolymph", "They are located in the vestibule"], 1,
  "Outer hair cells act as amplifiers and sharpen tuning."),
 ("The semicircular canals are mainly involved in...",
  ["Hearing", "Detecting head motion / balance", "Pressure equalisation", "Cerumen production"], 1,
  "They contain fluid and hair cells for head-motion detection."),
 ("The organ of Corti is located in the...",
  ["Scala vestibuli", "Scala media", "Scala tympani", "Vestibule"], 1,
  "It sits on the basilar membrane inside the scala media."),
],
"M7": [
 ("In the cochlea, high frequencies are represented at the...",
  ["Apex", "Base", "Middle only", "Round window membrane only"], 1,
  "Base = high frequencies (up to ~20,000 Hz); apex = low (~20 Hz)."),
 ("The basilar membrane is...",
  ["Widest and stiffest at the base", "Narrow and stiff at the base, wide and floppy at the apex", "Equally stiff throughout", "Narrow and floppy at the base"], 1,
  "This mechanical gradient underlies the tonotopic map."),
 ("In which direction does the traveling wave move?",
  ["From apex to base", "From base toward apex", "Side to side only", "It does not move"], 1,
  "It starts near the oval window (base) and moves toward the apex, peaking at a frequency-dependent place."),
 ("What happens to hair cells at the place of maximum traveling-wave displacement?",
  ["They are deformed and depolarise, firing the attached nerve fibres", "They stop responding", "They secrete wax", "They change into inner hair cells"], 0,
  "Deformation → depolarisation → auditory nerve firing."),
 ("A low-frequency tone (e.g., 100 Hz) produces its peak displacement near the...",
  ["Base", "Apex", "Oval window", "Round window"], 1,
  "Low frequencies peak near the floppy apex."),
],
"M8": [
 ("Auditory information from the auditory nerve first enters the CNS at the...",
  ["Cerebellum", "Medulla of the brainstem", "Frontal lobe", "Spinal cord"], 1,
  "From the auditory portion of CN VIII to the brainstem (medulla), then up to the cortex."),
 ("Primary auditory cortex is located in the...",
  ["Occipital lobe", "Temporal lobe", "Frontal lobe", "Parietal lobe only"], 1,
  "Temporal lobe, with auditory association areas for further analysis."),
 ("Why do humans have very precise binaural abilities?",
  ["Each ear projects to only one hemisphere", "Each ear is represented on both sides with extensive interactions between the two halves of the CNS", "Because the pinnae are large", "Because the cochlea has 5 turns"], 1,
  "Bilateral representation and cross-talk allow comparison of the two ears' signals."),
 ("On an audiogram, a larger dB HL number means...",
  ["Better hearing", "Worse hearing (higher threshold)", "A louder tone is played to the other ear", "No change"], 1,
  "The audiogram is plotted with 0 dB HL near the top – lower on the graph is a poorer threshold."),
 ("A 'rising' (low-frequency) loss shows...",
  ["Poorer thresholds at 250–500 Hz, improving at higher frequencies", "Poorer thresholds at 4000–8000 Hz", "Perfectly flat 0 dB HL line", "No thresholds at all"], 0,
  "The curve climbs toward normal as frequency increases."),
 ("Which set lists the standard octave test frequencies?",
  ["50, 100, 200, 400", "250, 500, 1000, 2000, 4000, 8000 Hz", "1, 2, 3, 4, 5, 6 Hz", "10,000–20,000 Hz only"], 1,
  "Standard octave frequencies 250–8000 Hz."),
],
}

# ---------------------------------------------------------------------------
# FLASHCARDS (term, definition, deck)
# ---------------------------------------------------------------------------
CARDS = [
 ("Temporal bone", "Complexly shaped paired skull bone that encases most peripheral auditory structures; contains the petrous part.", "Anatomy basics"),
 ("Petrous part of temporal bone", "Extremely hard, medial portion of the temporal bone that houses and protects the inner ear.", "Anatomy basics"),
 ("External auditory meatus", "Ear canal: ~2.5 cm tube open at the concha and closed at the eardrum; cartilage then bone about half-way.", "External ear"),
 ("Internal auditory meatus", "Opening on the medial surface of the petrous bone that carries CN VII and VIII toward the brainstem.", "Anatomy basics"),
 ("Pinna", "Cartilaginous outer ear; helps localise sound up-down and front-back.", "External ear"),
 ("Cerumen", "Ear wax made by glands in the cartilaginous canal; lubricates and protects.", "External ear"),
 ("Quarter-wavelength rule", "A tube closed at one end resonates at f ≈ c / 4L; for 2.5 cm this is ≈ 3000–3500 Hz.", "External ear"),
 ("Tympanic membrane", "Eardrum; three layers (two epithelial + middle fibrous/collagen); boundary between outer and middle ear.", "Tympanic membrane"),
 ("Pars flaccida", "Small slack part of the eardrum with only the two epithelial layers.", "Tympanic membrane"),
 ("Pars tensa", "Larger taut part of the eardrum with all three layers including the fibrous layer.", "Tympanic membrane"),
 ("Umbo / cone of light (beyond slides)", "Tip of the malleus handle at the centre of the drum / bright light reflex on the lower front part of a normal drum.", "Tympanic membrane"),
 ("Malleus", "Hammer; handle attached to the tympanic membrane.", "Middle ear"),
 ("Incus", "Anvil; middle ossicle forming joints with both the malleus and the stapes.", "Middle ear"),
 ("Stapes", "Stirrup; smallest bone in the body; oval footplate sits in the oval window.", "Middle ear"),
 ("Tensor tympani", "Middle-ear muscle (CN V) attaching to the malleus handle; contraction stiffens the ossicular chain.", "Middle ear"),
 ("Stapedius", "Middle-ear muscle (CN VII) attaching to the stapes; key effector of the acoustic reflex.", "Middle ear"),
 ("Epitympanum / mesotympanum / hypotympanum", "Upper / middle / lowest chambers of the middle ear (top, level with the drum, below).", "Middle ear"),
 ("Auditory (Eustachian) tube", "Connects middle ear to nasopharynx; tympanic end always open, pharyngeal end opens intermittently; equalises pressure.", "Middle ear"),
 ("Promontory", "Bulge on the medial wall of the middle ear formed by the cochlea; between oval and round windows.", "Middle ear"),
 ("Annular ligaments", "Tympanic annulus (holds the drum in its bony ring) and the annular ligament of the stapes (holds the footplate in the oval window).", "Middle ear"),
 ("Impedance mismatch", "Air (low impedance) to fluid (high impedance) interface would reflect most sound; the middle ear matches impedances.", "Impedance"),
 ("Middle-ear gain", "≈ 20–25 dB, mainly from the eardrum-to-footplate area ratio plus ossicular lever action.", "Impedance"),
 ("Bony labyrinth (otic capsule)", "Bony inner-ear capsule containing the semicircular canals, vestibule and cochlea.", "Inner ear"),
 ("Semicircular canals", "Detect head motion; important for balance.", "Inner ear"),
 ("Vestibule", "Detects head position and motion relative to the body.", "Inner ear"),
 ("Cochlea", "Spiral (2½ turns) structure with fluid and hair cells; detects sound.", "Inner ear"),
 ("Scala vestibuli / scala tympani", "Perilymph-filled ducts connected at the helicotrema (apex).", "Inner ear"),
 ("Scala media (cochlear duct)", "Endolymph-filled duct containing the organ of Corti; between Reissner's and basilar membranes.", "Inner ear"),
 ("Reissner's membrane", "Separates scala vestibuli from scala media.", "Inner ear"),
 ("Basilar membrane", "Separates scala media from scala tympani; supports the organ of Corti; narrow/stiff at base, wide/floppy at apex.", "Inner ear"),
 ("Tectorial membrane", "Membrane over the hair cells; linked to the hair-cell stereocilia.", "Hair cells"),
 ("Inner hair cells", "One row; primary frequency analysis; afferent auditory nerve fibres.", "Hair cells"),
 ("Outer hair cells", "Three rows; amplify and sharpen frequency analysis; efferent fibres.", "Hair cells"),
 ("Tonotopy", "Orderly map of frequency along the cochlea: high at base (~20,000 Hz), low at apex (~20 Hz).", "Hair cells"),
 ("Traveling wave", "Wave on the basilar membrane that builds over time from base toward apex and peaks at a frequency-specific place.", "Hair cells"),
 ("Binaural hearing", "Precise two-ear processing made possible by bilateral representation and cross-talk in the central pathways.", "Central & clinical"),
 ("Primary auditory cortex", "Located in the temporal lobe; receives the signal after brainstem pathways; association areas process further.", "Central & clinical"),
 ("Audiogram frequencies", "250, 500, 1000, 2000, 4000, 8000 Hz (plus 125 Hz and inter-octaves if needed).", "Central & clinical"),
 ("Low-frequency hearing loss", "Audiogram with poorer thresholds at 250–500 Hz and better at higher frequencies – a rising configuration.", "Central & clinical"),
]
print(f"Loaded: {len(EX)} practice exercises, {sum(len(v) for v in QUIZ.values())} quiz questions, {len(CARDS)} flashcards.")

---
# Module 1 · The temporal bone
**🎯 Goals:** describe where the temporal bone is, why it matters for hearing, and the roles of the two auditory meatuses.

### Key ideas (slides 2–5)
* The **temporal bone** is a complexly shaped, paired bone on each side of the skull. It **encases most of the peripheral auditory system**. Exceptions: the *cartilaginous* part of the outer ear and the *short stretch of auditory-vestibular nerve* between the internal auditory meatus and the brainstem.
* Seen from the side, it borders other skull bones and contains the bony **external auditory meatus**. In a top-down view (top of skull removed) it reaches **into the middle of the skull**.
* The more medial part is the **petrous part** – *extremely hard* bone that **contains and protects the inner ear**.
* Two openings matter most: the **external** auditory meatus (lateral surface, the entrance to the ear canal) and the **internal** auditory meatus (medial surface, the doorway for the nerves between inner ear and brainstem – cranial nerves **VII** and **VIII**).

### Memory trick
**"Petrous = Precious cargo."** The hardest bone in the skull wraps the most delicate sense organ.

> ⚠️ **Check with your instructor:** slide 4 says cranial nerves VI–XII all pass through the temporal bone. Standard anatomy texts usually name **VII and VIII** as the nerves entering the internal auditory meatus, and some of the others leave the skull through different openings. For hearing, remember VII + VIII.

In [ ]:
quiz('M1')

In [ ]:
practice_set(["6-1", "6-2"])

---
# Module 2 · Overview of the peripheral auditory system & the external ear
**🎯 Goals:** classify structures as conductive vs sensorineural; explain how the pinna and ear canal work.

### Three parts of the peripheral system (slides 6–7)
| Part | Filled with | Main structures |
|---|---|---|
| **External ear** | air | pinna, external auditory meatus, outer face of the tympanic membrane |
| **Middle ear** | air | tympanic cavity, ossicles, two muscles, ligaments, auditory-tube opening |
| **Inner ear** | **fluid** | cochlea, vestibule, semicircular canals |

The outer/middle boundary (tympanic membrane) is **tilted – more diagonal than vertical**.

**Conductive mechanism** = pinna → ear canal → tympanic membrane → middle-ear ossicles (carries sound mechanically).
**Sensorineural mechanism** = cochlea (hair cells) + auditory nerve (turns vibration into neural signals).

### Pinna (slide 8)
Cartilage with typical landmarks (exact shape varies from person to person, even in healthy ears). The cartilage continues part-way into the ear canal. Its folds help us **localise sounds up/down and front/back**.

### External auditory meatus (slides 9–10)
* Tube **open** at the concha, **closed** by the eardrum; typical length **≈ 2.5 cm**.
* First half **cartilage** (continuation of the pinna), then **bone** (temporal bone). It is **bent** and slopes gently upward toward the drum.
* **Resonance:** for a tube closed at one end, the lowest resonant frequency has a wavelength four times the tube length ("¼-wavelength rule"): **f ≈ c / 4L ≈ 3000–3500 Hz**.
* **Protection:** the *kink* (bending toward the back of the head) shields the drum from foreign objects; glands in the cartilaginous part make **cerumen** (lubricates, resists bacteria/debris); the *downward slope* toward the concha lets dead skin cells slide away from the drum.

### 🧪 Lab: change the canal length and watch the resonance move

In [ ]:
ear_canal_lab()

In [ ]:
quiz('M2')

In [ ]:
practice_set(["6-4", "6-5"])

---
# Module 3 · The tympanic membrane (eardrum)
**🎯 Goals:** name the three layers, contrast pars tensa vs pars flaccida, recognise otoscopic landmarks.

### Key ideas (slide 11)
* The membrane is **tilted** in a coronal view.
* **Three tissue layers:** the outer layer (faces the ear canal) and the inner layer (faces the middle ear) are **epithelial**; the **middle layer is fibrous, collagenous tissue**.
* The **middle (fibrous) layer is exquisitely sensitive to molecule-sized pressure variations.**
* A small region – the **pars flaccida** – has *only* the two epithelial layers. The rest (the larger, taut **pars tensa**) has all three.
* In an otoscopic view a healthy membrane is **translucent**, so landmarks show through.

### 🔎 Beyond the slides – otoscopic landmarks
| Landmark | What it is |
|---|---|
| **Handle (manubrium) of malleus** | pale streak visible through the drum – the malleus is attached here |
| **Umbo** | tip of the handle, near the centre |
| **Short (lateral) process** | small bump at the top end of the handle |
| **Cone of light** | triangular reflection of the otoscope light, front-lower part of the drum |
| **Pars flaccida / pars tensa** | upper slack part / rest of the drum |

**Right vs left ear:** cone of light about **5 o'clock = right ear**, **7 o'clock = left ear**.

### Memory trick
*Tensa = Tense (tight, 3 layers). Flaccida = Flaccid (floppy, 2 layers).*

In [ ]:
quiz('M3')

In [ ]:
practice_set(["6-6", "6-7", "6-8"])

---
# Module 4 · The middle ear cavity
**🎯 Goals:** name the contents and chambers, ossicles, joints, muscles, ligaments, the auditory tube and key landmarks.

### Contents (slide 12)
Three **ossicles**, two **muscles**, several **ligaments**, and the **open end of the auditory tube**.

### Three chambers (slide 13)
| Chamber | Position | Contents |
|---|---|---|
| **Epitympanum** | above the drum | head of malleus, body of incus, antrum |
| **Mesotympanum** | level with the drum (top-to-bottom edge) | main chamber |
| **Hypotympanum** | lowest | no functional middle-ear parts |

Boundaries are arbitrary but useful for communication.

### Ossicles (slide 14) – lateral → medial
1. **Malleus** (hammer) – *handle attached to the tympanic membrane*.
2. **Incus** (anvil) – middle bone; **forms joints with both others**.
3. **Stapes** (stirrup) – **smallest bone in the body**; oval **footplate** fits the **oval window** of the bony labyrinth.

Joints: **incudomalleolar** and **incudostapedial**. The ossicular chain has a resonance near **1400 Hz**.

### Ligaments & muscles (slide 15)
| Muscle | Attaches to | Cranial nerve | Role |
|---|---|---|---|
| **Tensor tympani** | handle of malleus | **V** | contraction **stiffens** the ossicular chain |
| **Stapedius** | stapes | **VII** | pulls on the footplate in very intense sound; key to the **acoustic reflex**; stops the footplate being driven too far into the cochlear fluid |

Ligaments hold the ossicles and limit joint and footplate movement.

### 🔎 Beyond the slides – walls, tube, landmarks
* **Lateral wall** = tympanic membrane (*membranous wall*). **Medial wall** = bony *labyrinthine wall*, with the **promontory** (bulge made by the cochlea) between the **oval** and **round** windows.
* **Auditory (Eustachian) tube:** tympanic end always open; pharyngeal end normally closed, opens when we swallow/yawn → **equalises pressure** so the drum can move freely.
* **Annular ligaments:** the tympanic annulus (rim of the drum) and the annular ligament of the stapes (ring around the footplate).
* **Facial nerve (VII)** runs through the bony wall of the cavity.

### Memory tricks
* **M-I-S** = Malleus → Incus → Stapes (outside → inside).
* **S**tapedius → **S**tapes. The *other* muscle (Tensor tympani) → the other bone attached to the drum, the Malleus.
* **E-M-H** from top: Epi, Meso, Hypo.

In [ ]:
quiz('M4')

### ✍️ Revisit the worksheet figure (exercise 6-3) – grab your printed worksheet

In [ ]:
practice_set(["6-3a", "6-3b", "6-3c", "6-3d", "6-3e", "6-3f", "6-3g"])

### ✍️ Middle-ear exercises

In [ ]:
practice_set(["6-9", "6-10", "6-11", "6-12", "6-13", "6-14", "6-15", "6-16"])

---
# Module 5 · Impedance mismatch & middle-ear gain
**🎯 Goals:** explain why we need a middle ear, and where the 20–25 dB gain comes from.

### The problem
**Impedance** = how strongly a medium resists being set in motion. Air is easy to move (**low impedance**); the cochlear fluid is hard to move (**high impedance**). If airborne sound struck the fluid directly, **most of the energy would bounce back** (≈ 99 %, roughly a 30 dB loss).
*Analogy: shouting to a swimmer under water – most of your voice reflects off the surface.*

### The solution: the middle ear as a transformer
1. **Area ratio (main factor).** The effective vibrating area of the eardrum is much larger than the stapes footplate (≈ 17 : 1 in typical textbook values). Same force on a smaller area ⇒ **higher pressure**: pressure gain = 20·log₁₀(area ratio) ≈ **25 dB** by itself.
2. **Lever action.** The malleus handle is longer than the incus' long process (≈ 1.3 : 1) ⇒ a couple more dB.
3. In living ears the measured net gain is about **20–25 dB** (the drum is not a perfectly rigid piston and some energy is lost).

*Numbers above are typical textbook values (beyond the slides), used so you can experiment.*

### 🧪 Lab: break the middle ear
Try shrinking the eardrum area or making the footplate bigger and think about what that would mean for a patient (e.g., eardrum perforation, otosclerosis).

In [ ]:
middle_ear_gain_lab()

In [ ]:
quiz('M5')

In [ ]:
practice_set(["6-17", "6-18"])

---
# Module 6 · The inner ear: bony labyrinth, cochlea and organ of Corti
**🎯 Goals:** identify the parts of the bony labyrinth, the cochlear ducts/membranes/fluids, and the hair cells.

### Bony labyrinth (otic capsule) – slide 17
Two openings in the bone: **oval window** (stapes footplate) and **round window**. From back to front:
| Structure | Function |
|---|---|
| **Semicircular canals** | fluid + hair cells detect **head motion** → balance |
| **Vestibule** | detects **head position** and motion relative to the body |
| **Cochlea** | fluid + hair cells detect **sound** |

### Cochlea (slide 18)
* Spiral of **2½ turns** from base to apex.
* **Three fluid-filled ducts:**

| Duct | Fluid | Separated from next by |
|---|---|---|
| **Scala vestibuli** (top) | perilymph | **Reissner's membrane** |
| **Scala media / cochlear duct** (middle) | **endolymph** | **basilar membrane** |
| **Scala tympani** (bottom) | perilymph | — |

* Scala vestibuli and scala tympani join at the **apex (helicotrema)**.
* The **organ of Corti** sits in the **scala media**.

### Organ of Corti (slides 19–20)
* The **basilar membrane** (stiffer than Reissner's) supports it.
* **One row of inner hair cells, three rows of outer hair cells**, running base to apex.
* **Stereocilia** project from the hair-cell tops; the **tectorial membrane** lies above (linked to inner hair cells by fluid motion, directly to the outer hair cells).
* Nerve fibres leave the base of the hair cells: **inner** → *afferent* (to brain) part of the auditory nerve; **outer** → *efferent* part.
* **Inner hair cells:** primary **frequency analysis**. **Outer hair cells:** **amplify** and **sharpen** that analysis.

### Memory tricks
* **"I" = Information** (inner cells send the main message); **"O" = Octane booster** (outer cells boost and sharpen).
* **Top-to-bottom: V-M-T** = vestibuli, media, tympani.

In [ ]:
quiz('M6')

In [ ]:
practice_set(["6-19", "6-20", "6-21", "6-22"])

---
# Module 7 · Tonotopy and the traveling wave
**🎯 Goals:** explain how the cochlea sorts sound by frequency.

### Mechanical gradient (slide 21)
The basilar membrane is **narrow and stiff at the base** and **wide and floppy at the apex**. Fluid pushed in at the **oval window** flows through the scala vestibuli, around the helicotrema, and out toward the **round window** (scala tympani).
*Think of a harp: short tight strings (high notes) at one end, long loose strings (low notes) at the other.*

### Traveling wave (slide 22)
* The stapes footplate pushes the cochlear fluid; the pattern of motion mirrors the frequency/amplitude of the sound.
* These are **traveling waves** – they develop over time along the basilar membrane.
* The wave displaces the membrane and deforms the hair cells → **depolarisation** → **auditory nerve fires**.
* The **peak** of the wave shows the dominant frequency: **high frequency → peak near the base; low frequency → peak near the apex**.

### Tonotopic map (slide 20)
**≈ 20,000 Hz at the base → ≈ 20 Hz at the apex.** Both inner and outer hair cells follow this map.

### 🧪 Lab: where does my tone peak?
(Uses the Greenwood frequency–position function; wave shape is schematic.)

In [ ]:
tonotopy_lab()

In [ ]:
quiz('M7')

In [ ]:
practice_set(["6-23", "6-24"])

---
# Module 8 · Central pathways, binaural hearing and the audiogram
**🎯 Goals:** outline the route to the cortex, explain binaural precision, and read a simple audiogram.

### Auditory pathways (slides 23–24)
* From the auditory part of **cranial nerve VIII**, dedicated pathways carry the signal from its CNS entry point (**medulla of the brainstem**) to the **primary auditory cortex** (temporal lobe), with further analysis in **auditory association areas**.
* **Each ear is represented on both sides** of the brain, with **extensive interactions between the two halves of the CNS** → **high-precision binaural abilities** (e.g., localisation).

🔎 *Beyond the slides:* a typical route is auditory nerve → cochlear nucleus → superior olivary complex (first major binaural comparison) → lateral lemniscus → inferior colliculus → medial geniculate body (thalamus) → primary auditory cortex.

### The audiogram (practice exercises 6-26, 6-27)
* Tests pure-tone **thresholds** at **250, 500, 1000, 2000, 4000 and 8000 Hz** (often 125 Hz and 3000/6000 Hz too).
* **Frequency** on the horizontal axis (low → high); **hearing level in dB HL** on the vertical axis, with 0 at the **top** – so a point that falls **lower** means a **poorer** threshold.
* **Low-frequency loss ⇒ "rising" curve:** worse at 250–500 Hz, improving toward normal at higher frequencies.

### 🧪 Lab: build an audiogram
Choose a preset or drag the sliders, then describe the pattern in words.

In [ ]:
audiogram_lab()

In [ ]:
quiz('M8')

In [ ]:
practice_set(["6-25", "6-26", "6-27"])

---
# 🏁 Wrap-up
### 🃏 Flashcards
Choose a deck by editing the name in quotes (decks: `"Anatomy basics"`, `"External ear"`, `"Tympanic membrane"`, `"Middle ear"`, `"Impedance"`, `"Inner ear"`, `"Hair cells"`, `"Central & clinical"`) or use `None` for everything.

In [ ]:
flashcards(deck=None)

### 🎓 Final mixed exam (20 random questions – run the cell again for a new set)

In [ ]:
final_exam(20)

### 📊 Scoreboard

In [ ]:
scoreboard()

### 🖨️ Answer key (all 27 exercises)
Run this cell, then use *File → Print* if you want a paper copy.

In [ ]:
answer_key()

---
### Study tips
* **Retrieve, don't re-read:** always type your own answer *before* revealing the model answer.
* **Space it out:** repeat the quizzes after 1 day, 3 days and 1 week.
* **Draw it:** sketch the ear (outer → middle → inner) and the cochlear cross-section from memory, then check against your slides.
* **Teach it:** explain impedance matching and tonotopy aloud to a friend in under a minute each.

*Tutor notebook created from the course deck and Chapter 6 practice exercises. Always defer to your instructor's wording on exams.*